# GeoInsight - Member A Validation (Geospatial Data & Processing)

This notebook validates the Member-A processing pipeline for:

- **District:** Kamrup, Assam, India
- **Period:** June 2026
- **Metrics:** Sentinel-2 mean NDVI, CHIRPS total June rainfall, JRC water coverage

JRC Global Surface Water v1.4 is historical surface-water occurrence through 2021; the water metric is a reference metric, not a June 2026 observation.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "boundaries" / "kamrup.geojson").exists():
            return candidate
    return path

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"

BOUNDARY_PATH = DATA_DIR / "boundaries" / "kamrup.geojson"
RASTERS = {
    "sentinel2_b4": DATA_DIR / "sentinel2" / "sentinel2_B04_june2026.tif",
    "sentinel2_b8": DATA_DIR / "sentinel2" / "sentinel2_B08_june2026.tif",
    "rainfall": DATA_DIR / "rainfall" / "chirps_june2026.tif",
    "water": DATA_DIR / "water" / "jrc_water.tif",
}
print(f"Project root: {PROJECT_ROOT}")
print(f"Boundary: {BOUNDARY_PATH}")
print(f"Boundary exists: {BOUNDARY_PATH.exists()}")

## 1. Load the Kamrup boundary and verify it

In [ ]:
boundary = gpd.read_file(BOUNDARY_PATH)
print(f"features: {len(boundary)}")
print(f"CRS: {boundary.crs}")
print(f"name: {boundary.iloc[0]['Name'] if 'Name' in boundary.columns else 'n/a'}")
print(f"geometry: {boundary.geometry.iloc[0].geom_type}")

## 2. Check raster availability

Missing files are reported and never fabricated.

In [ ]:
for label, path in RASTERS.items():
    if path.exists():
        print(f"[OK] {label}: {path}")
    else:
        print(f"[MISSING] {path}")

all_present = BOUNDARY_PATH.exists() and all(p.exists() for p in RASTERS.values())
print("\nAll datasets present:", all_present)

## 3. NDVI (Sentinel-2 B4/B8, June 2026)

In [ ]:
from app.processing import ndvi

if RASTERS["sentinel2_b4"].exists() and RASTERS["sentinel2_b8"].exists():
    mean_ndvi = ndvi.calculate_ndvi(
        RASTERS["sentinel2_b4"], RASTERS["sentinel2_b8"], BOUNDARY_PATH
    )
    print(f"Mean NDVI: {mean_ndvi:.4f}")
else:
    mean_ndvi = None
    print("NDVI: skipped (Sentinel-2 raster missing)")

## 4. Rainfall (CHIRPS, June 2026 total)

In [ ]:
from app.processing import rainfall

if RASTERS["rainfall"].exists():
    total_rainfall = rainfall.calculate_monthly_rainfall(
        RASTERS["rainfall"], BOUNDARY_PATH
    )
    print(f"June 2026 total rainfall: {total_rainfall:.2f} mm")
else:
    total_rainfall = None
    print("Rainfall: skipped (CHIRPS raster missing)")

## 5. Water (JRC Global Surface Water, historical reference)

In [ ]:
from app.processing import water

if RASTERS["water"].exists():
    water_result = water.calculate_water_area(RASTERS["water"], BOUNDARY_PATH)
    for key, value in water_result.items():
        print(f"{key}: {value}")
else:
    water_result = None
    print("Water: skipped (JRC raster missing)")

## 6. Final results

In [ ]:
print("=" * 40)
print("GEOINSIGHT - MEMBER A VALIDATION")
print("=" * 40)
print("District: Kamrup")
print("State: Assam")
print("Period: June 2026")
print()

if mean_ndvi is not None:
    print("NDVI:")
    print(f"Mean NDVI: {mean_ndvi:.4f}")
    print()

if total_rainfall is not None:
    print("Rainfall:")
    print(f"June 2026 total rainfall: {total_rainfall:.2f} mm")
    print()

if water_result is not None:
    print("Water:")
    print(f"Water pixels: {water_result['water_pixels']}")
    print(f"Valid pixels: {water_result['valid_pixels']}")
    print(f"Water percentage: {water_result['water_percentage']:.2f} %")
    print(f"Water area: {water_result['water_area_km2']:.2f} km2")